# 📐 Floor Plan CAD Refiner via Reasoning LLMs (Google Colab T4/L4/A100)

This notebook takes the multi-sensor intermediate representation generated by your local pipeline:
- **YOLOv8-seg**: Semantic bounding boxes (rooms, doors, windows)
- **OpenCV LSD**: Sub-pixel wall vectors & orthogonal grid axes
- **Tesseract OCR**: Ground-truth dimensions ('14.50m total', '5.00m x 4.00m')

And feeds it into an advanced **Frontier Reasoning LLM** (`DeepSeek-R1-Distill-Qwen-8B` or `Qwen2.5-Coder-7B`) running directly inside Colab in 4-bit precision to generate clean, production-ready vector CAD / SVG code.


In [ ]:
# 🚀 Step 1: Install Dependencies
!pip install -q transformers bitsandbytes accelerate
print('✅ Libraries installed successfully!')


In [ ]:
# 🔍 Step 2: Verify GPU Acceleration
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Active GPU:', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'Total VRAM: {vram:.2f} GB')
else:
    print('⚠️ WARNING: GPU not detected! Go to Runtime -> Change runtime type -> Select T4 GPU.')


In [ ]:
# 🧠 Step 3: Load Reasoning Model in 4-bit NF4
# Uses only ~5.5 GB VRAM (Fits easily on Free Colab T4 15GB GPU)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Model Selection:
# Option 1 (Default): DeepSeek-R1-Distill-Qwen-8B (Elite chain-of-thought architectural reasoning)
# Option 2: Qwen/Qwen2.5-Coder-7B-Instruct (Precise CAD / SVG code generation)
model_id = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-8B'

print(f'⏳ Downloading & Loading {model_id} in 4-bit...')

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map='auto'
)

print(f'✅ Model {model_id} loaded and ready!')


In [ ]:
# 📄 Step 4: Load the Master Prompt
# You can upload your 'gemini_reasoning_prompt.md' using Colab's file sidebar,
# or run this cell to select and upload it directly.

import os
from google.colab import files

prompt_path = 'gemini_reasoning_prompt.md'

if not os.path.exists(prompt_path):
    print('📁 Please upload your gemini_reasoning_prompt.md file:')
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.md'):
            prompt_path = fn
            break

with open(prompt_path, 'r', encoding='utf-8') as f:
    prompt_text = f.read()

print(f'✅ Loaded prompt from {prompt_path} ({len(prompt_text):,} characters)!')


In [ ]:
# ⚡ Step 5: Run Architectural Reasoning & SVG Generation
import time

print('🧠 Thinking through architectural constraints & generating CAD plan...')
start_time = time.time()

messages = [{'role': 'user', 'content': prompt_text}]
prompt_formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt_formatted, return_tensors='pt').to('cuda')

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=3000,
        temperature=0.2,
        top_p=0.95,
        do_sample=True,
    )

output_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
elapsed = time.time() - start_time
print(f'✨ Finished generation in {elapsed:.1f} seconds!')


In [ ]:
# 🎨 Step 6: Render & Save Vector Floor Plan
import re
from IPython.display import display, HTML
from google.colab import files

svg_match = re.search(r'(<svg[\s\S]*?</svg>)', output_text)

if svg_match:
    svg_code = svg_match.group(1)
    output_filename = 'refined_cad_floorplan.svg'
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write(svg_code)
    print(f'💾 Saved SVG to: {output_filename}')
    print('👇 Rendered Vector CAD Floor Plan:')
    display(HTML(f'<div style="background:#fff; padding:20px; border-radius:8px; border:1px solid #e2e8f0;">{svg_code}</div>'))
    
    # Download automatically to your computer
    files.download(output_filename)
else:
    print('⚠️ Could not extract <svg> tag. Printing full model response:\n')
    print(output_text)
